>важно заранее определить, какие свойства и действия должны быть у этих объектов

### Процедурное программирование

In [ ]:
def create_car(color, consumption, tank_volume, mileage=0):
    return {
        "color": color,
        "consumption": consumption,
        "tank_volume": tank_volume,
        "reserve": tank_volume,
        "mileage": mileage,
        "engine_on": False
    }

def start_engine(car):
    if not car["engine_on"] and car["reserve"] > 0:
        car["engine_on"] = True
        return "Двигатель запущен."
    return "Двигатель уже был запущен"

def stop_engine(car):
    if car["engine_on"]:
        car["engine_on"] = False
        return "Двигатель остановлен."
    return "Двигатель уже был остановлен."

def drive(car, distance):
   if not car["engine_on"]:
       return "Двигатель не запущен."
   if car["reserve"] / car["consumption"] * 100 < distance:
       return "Малый запас топлива."
   car["mileage"] += distance
   car["reserve"] -= distance / 100 * car["consumption"]
   return f"Проехали {distance} км. Остаток топлива: {car['reserve']} л."

def refuel(car):
   car["reserve"] = car["tank_volume"]

def get_mileage(car):
   return f"Пробег {car['mileage']} км."

def get_reserve(car):
   return f"Запас топлива {car['reserve']} л."

car_1 = create_car(color="black", consumption=10, tank_volume=55)

print(start_engine(car_1))
print(drive(car_1, 100))
print(drive(car_1, 100))
print(drive(car_1, 100))
print(drive(car_1, 300))
print(get_mileage(car_1))
print(get_reserve(car_1))
print(stop_engine(car_1))
print(drive(car_1, 100))

### ООП

Класс — это шаблон (или описание), по которому создаются объекты. В классе задаются свойства (атрибуты) и действия (методы), общие для всех объектов этого типа.

Объект — это конкретный экземпляр класса, обладающий собственными значениями атрибутов. Объект знает, к какому классу он принадлежит, и может выполнять описанные в нём действия.

Python построен на объектной модели. Это значит, что практически всё в нём — объекты, созданные на основе классов (числа, строки, функции, списки и даже модули)

In [ ]:
print(type(1))   # <class 'int'>
print(type("1")) # <class 'str'>

Согласно стандарту оформления PEP 8, имя класса записывается в стиле CapWords. По умолчанию Python хранит все атрибуты объекта в обычном словаре __dict__. Это делает объекты динамическими. (-память и -скорость)

`__slots__` оптимизирует объект, убирая скрытый словарь __dict__ и фиксируя структуру в памяти.

In [ ]:
# Современный production код

from datetime import datetime
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, computed_field

# BaseModel в Pydantic автоматически валидирует типы при создании и изменении
class Car(BaseModel):
    # Ограничиваем список разрешенных цветов на уровне типов (Literal)
    color: Literal["Black", "Red", "White"] = "Black"
    is_engine_running: bool = False
    year: int = Field(default=2000, ge=1900, le=datetime.now().year)

    # Запрещаем добавлять случайные атрибуты (аналог __slots__ / защита от опечаток)
    model_config = {"extra": "forbid", "validate_assignment": True}

    @computed_field  # Автоматически делает свойство динамическим и Read-only (как @property)
    def age(self) -> int:
        return datetime.now().year - self.year

    def start_engine(self) -> str:
        if self.is_engine_running:
            return "Двигатель уже заведен"
        
        self.is_engine_running = True
        return "Старт..."

if __name__ == "__main__":
    # Сценарий 1: Прием «грязных» данных из внешнего API/фронтенда
    raw_json_data = '{"color": "Red", "year": 2021, "is_engine_running": false}'

    # Pydantic сам парсит строку, проверяет типы и создает объект
    car_from_web = Car.model_validate_json(raw_json_data)
    print(
        f"Машина из сети: цвет {car_from_web.color}, возраст {car_from_web.age} лет"
    )

    # Сценарий 2: Полноценный JSON-ответ для фронтенда (включая вычисляемое поле age)
    # Метод model_dump_json() автоматически соберет структуру вместе с age
    print("\nГотовый JSON для отправки клиенту:")
    print(car_from_web.model_dump_json(indent=2))

    # Сценарий 3: Перехват детальной ошибки для логирования
    try:
        # Передаем заведомо некорректные данные
        bad_car = Car(color="Green", year=1850)
    except ValidationError as e:
        print("\nSenior-логирование ошибок (Pydantic выдает структурированный JSON):")
        # Вместо падения программы мы получаем понятный отчет, какое именно поле сломалось
        print(e.json(indent=2))

Допустим, мы хотим написать функцию `range_reserve()`, которая рассчитывает запас хода автомобиля. Она должна одинаково работать и с `Car`, и с `ElectricCar`. Такое поведение называется `полиморфизмом` — когда один и тот же интерфейс (в нашем случае — функция) работает с объектами разных типов.

Чтобы это было возможно, в обоих классах должны быть методы с одинаковыми именами и сигнатурами. Для расчёта запаса хода нам понадобится ещё один метод — `get_consumption():`

In [ ]:
# Учебный код

class Car:

    def __init__(self, color, consumption, tank_volume, mileage=0):
        self.color = color
        self.consumption = consumption
        self.tank_volume = tank_volume
        self.reserve = tank_volume
        self.mileage = mileage
        self.engine_on = False

    def start_engine(self):
        if not self.engine_on and self.reserve > 0:
            self.engine_on = True
            return "Двигатель запущен."
        return "Двигатель уже был запущен."

    def stop_engine(self):
        if self.engine_on:
            self.engine_on = False
            return "Двигатель остановлен."
        return "Двигатель уже был остановлен."

    def drive(self, distance):
        if not self.engine_on:
            return "Двигатель не запущен."
        if self.reserve / self.consumption * 100 < distance:
            return "Малый запас топлива."
        self.mileage += distance
        self.reserve -= distance / 100 * self.consumption
        return f"Проехали {distance} км. Остаток топлива: {self.reserve} л."

    def refuel(self):
        self.reserve = self.tank_volume

    def get_mileage(self):
        return self.mileage

    def get_reserve(self):
        return self.reserve

    def get_consumption(self):
        return self.consumption


class ElectricCar:

    def __init__(self, color, consumption, bat_capacity, mileage=0):
        self.color = color
        self.consumption = consumption
        self.bat_capacity = bat_capacity
        self.reserve = bat_capacity
        self.mileage = mileage
        self.engine_on = False

    def start_engine(self):
        if not self.engine_on and self.reserve > 0:
            self.engine_on = True
            return "Двигатель запущен."
        return "Двигатель уже был запущен."

    def stop_engine(self):
        if self.engine_on:
            self.engine_on = False
            return "Двигатель остановлен."
        return "Двигатель уже был остановлен."

    def drive(self, distance):
        if not self.engine_on:
            return "Двигатель не запущен."
        if self.reserve / self.consumption * 100 < distance:
            return "Малый заряд батареи."
        self.mileage += distance
        self.reserve -= distance / 100 * self.consumption
        return f"Проехали {distance} км. Остаток заряда: {self.reserve} кВт*ч."

    def recharge(self):
        self.reserve = self.bat_capacity

    def get_mileage(self):
        return self.mileage

    def get_reserve(self):
        return self.reserve

    def get_consumption(self):
        return self.consumption


def range_reserve(car):
    return car.get_reserve() / car.get_consumption() * 100


car_1 = Car(color="black", consumption=10, tank_volume=55)
car_2 = ElectricCar(color="white", consumption=15, bat_capacity=90)
print(f"Запас хода: {range_reserve(car_1)} км.")
print(f"Запас хода: {range_reserve(car_2)} км.")


In [ ]:
class Point:

    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def move(self, dx, dy):
        self.x += dx
        self.y += dy
    
    def length(self, other):
        return round(((self.x - other.x)**2 + (self.y - other.y)**2)**0.5, 2)


point = Point(3, 5)
print(point.x, point.y)
point.move(2, -3)
print(point.x, point.y)


first_point = Point(2, -7)
second_point = Point(7, 9)
print(first_point.length(second_point))
print(second_point.length(first_point))


In [ ]:
class RedButton:

    def __init__(self, message="Тревога!", counter=0):
        self.message = message
        self.counter = counter

    def click(self):
        self.counter += 1
        print(self.message)
    
    def count(self):
        return self.counter

first_button = RedButton()
second_button = RedButton()
for time in range(5):
    if time % 2 == 0:
        second_button.click()
    else:
        first_button.click()
print(first_button.count(), second_button.count())

In [ ]:
class Programmer:

    def __init__(self, name, position):
        self.name = name
        self.position = position
        self.hours_worked = 0
        self.money = 0

        if self.position == "Junior":
            self.salary = 10
        elif self.position == "Middle":
            self.salary = 15
        elif self.position == "Senior":
            self.salary = 20

    def work(self, time):
        self.hours_worked += time
        self.money += time * self.salary
    
    def rise(self):
        if self.position == "Junior":
            self.position = "Middle"
            self.salary = 15
        elif self.position == "Middle":
            self.position = "Senior"
            self.salary = 20
        elif self.position == "Senior":
            self.salary += 1
    
    def info(self):
        return f"{self.name} {round(self.hours_worked)}ч. {self.money}тгр."


programmer = Programmer('Васильев Иван', 'Junior')
programmer.work(750)
print(programmer.info())
programmer.rise()
programmer.work(500)
print(programmer.info())
programmer.rise()
programmer.work(250)
print(programmer.info())
programmer.rise()
programmer.work(250)
print(programmer.info())

### Наследование

In [6]:
class Pencil:

    def __init__(self, color="серый"):
        self.color = color

    def draw_picture(self):
        return f"Нарисован рисунок цветом '{self.color}'."

class Pen(Pencil):

    def sign_document(self):
        if self.color not in ("синий", "чёрный", "фиолетовый"):
            return f"Ручкой цвета '{self.color}' нельзя подписать документ."
        return f"Подписан документ."

pen = Pen()

print(pen.draw_picture())
print(pen.sign_document())
blue_pen = Pen(color="синий")
print(blue_pen.sign_document())


Нарисован рисунок цветом 'серый'.
Ручкой цвета 'серый' нельзя подписать документ.
Подписан документ.


### Множественное наследование  

- вы хотите объединить поведение из нескольких независимых классов;
- базовые классы реализуют разные аспекты (например, приветствие, логирование, сохранение данных);
- необходимо повторно использовать код без создания глубокой иерархии.

In [ ]:
class GreetingFormal:

    def __init__(self):
        self.formal_greeting = "Добрый день,"

    def greet_formal(self, name):
        return f"{self.formal_greeting} {name}!"

class GreetingInformal:

    def __init__(self):
        self.informal_greeting = "Привет,"

    def greet_informal(self, name):
        return f"{self.informal_greeting} {name}!"

class GreetingMix(GreetingFormal, GreetingInformal):

    def __init__(self):
        # super().__init__() - подтянет только 1 родителя
        GreetingFormal.__init__(self)
        GreetingInformal.__init__(self)

mixed_greeting = GreetingMix()
print(mixed_greeting.greet_formal("Пользователь"))
print(mixed_greeting.greet_informal("Пользователь"))


Добрый день, Пользователь!
Привет, Пользователь!


### Method Resolution Order - MRO (порядок разрешения методов)  
Если в базовых классах есть совпадающие имена, приоритет получит тот класс, что указан `первым`. Поэтому при множественном наследовании важно чётко продумывать структуру и порядок инициализации.

In [10]:
class Car:

    def __init__(self, color, consumption, tank_volume, mileage=0):
        self.color = color
        self.consumption = consumption
        self.tank_volume = tank_volume
        self.reserve = tank_volume
        self.mileage = mileage
        self.engine_on = False

    def start_engine(self):
        if not self.engine_on and self.reserve > 0:
            self.engine_on = True
            return "Двигатель запущен."
        return "Двигатель уже был запущен."

    def stop_engine(self):
        if self.engine_on:
            self.engine_on = False
            return "Двигатель остановлен."
        return "Двигатель уже был остановлен."

    def drive(self, distance):
        if not self.engine_on:
            return "Двигатель не запущен."
        if self.reserve / self.consumption * 100 < distance:
            return "Малый запас топлива."
        self.mileage += distance
        self.reserve -= distance / 100 * self.consumption
        return f"Проехали {distance} км. Остаток топлива: {self.reserve} л."

    def refuel(self):
        self.reserve = self.tank_volume

    def get_mileage(self):
        return self.mileage

    def get_reserve(self):
        return self.reserve

    def get_consumption(self):
        return self.consumption

class ElectricCar(Car):

    def __init__(self, color, consumption, bat_capacity, mileage=0):
        super().__init__(color, consumption, bat_capacity, mileage)
        self.bat_capacity = bat_capacity

    def drive(self, distance):
        if not self.engine_on:
            return "Двигатель не запущен."
        if self.reserve / self.consumption * 100 < distance:
            return "Малый запас заряда."
        self.mileage += distance
        self.reserve -= distance / 100 * self.consumption
        return f"Проехали {distance} км. Остаток заряда: {self.reserve} кВт*ч."

    def recharge(self):
        self.reserve = self.bat_capacity
    
    def __str__(self):
        return f"Электромобиль. " \
            f"Цвет: {self.color}. " \
            f"Пробег: {self.mileage} км. " \
            f"Остаток заряда: {self.reserve} кВт*ч."


electric_car = ElectricCar(color="white", consumption=15, bat_capacity=90)
print(electric_car.start_engine())
print(electric_car.drive(100))

print(electric_car)

# Вывод программы:
# <__main__.ElectricCar object at 0x000002365DDD8A00>


Двигатель запущен.
Проехали 100 км. Остаток заряда: 75.0 кВт*ч.
Электромобиль. Цвет: white. Пробег: 100 км. Остаток заряда: 75.0 кВт*ч.


### Магические методы

Такие методы вызываются автоматически при выполнении встроенных операций и функций и позволяют настроить поведение объектов под нужды вашей программы.

Этот механизм называется `перегрузкой операторов (operator overloading)`. Вы можете определить, как именно ваш объект будет вести себя при сложении, сравнении, индексировании, отображении и других операциях.

---

Сводная шпаргалка по фреймворкам:

- **Пишете кастомный Dataset?** Реализуйте: `__len__`, `__getitem__`.
- **Пишете кастомный слой / блок сети / модуль инференса?** Реализуйте: `__call__`.
- **Пишете математический класс (вектор, bounding box, кастомный тензор)?** Реализуйте: `__matmul__`, `__add__`, `__iadd__`.

---

1. Математика и тензорные операции (Критически важные)

В ML вы постоянно будете писать обертки над тензорами, кастомные функции потерь (loss), аугментации или слои нейросетей.

- `__add__(self, other)` (+), `__sub__` (-), `__mul__` (*) — **Поэлементные операции**. Нужны для кастомной предобработки данных, применения масок к изображениям или масштабирования признаков.
- `__matmul__(self, other)` — **Матричное умножение (`@`)**. Это фундамент для написания кастомных слоев (например, линейных или слоев внимания / Attention), линейной алгебры и геометрических преобразований в Computer Vision.
- `__iadd__`, `__imul__` и др. — **In-place операции (`+=`, `*=`)**. В ML они критически важны для **оптимизации памяти**. Операция `x += y` не создает новый тензор (в отличие от `x = x + y`), что спасает от переполнения VRAM (видеопамяти) или RAM при работе с тяжелыми батчами картинок.

2. Пайплайны, слои и модели (Архитектурные)

- `__call__(self, *args, **kwargs)` — **Прямой проход (Forward Pass)**. В AI это самый используемый метод. Все слои и модели в _PyTorch_ (`nn.Module`) вызываются именно через `__call__` (который под капотом вызывает метод `forward`). Если вы пишете свой конвейер обработки данных (Data Pipeline) или инференс-сервис, этот метод сделает код идиоматичным: `output = model(image)`.

3. Датасеты, Батчинг и Слайсинг (Работа с данными)

Работа с датасетами (особенно в CV) — это постоянная нарезка (slicing) и индексация.

- `__getitem__(self, key)` — **Доступ к элементу / Слайсинг (`dataset[i]`)**. Основа для любого кастомного `Dataset` (в _PyTorch_ или _TensorFlow_). Позволяет на лету доставать картинку по индексу, применять к ней аугментации и отдавать в загрузчик батчей (`DataLoader`). Также используется для обрезки регионов на фото (crop) по координатам bounding box'а: `image_np[y1:y2, x1:x2]`.
- `__len__(self)` — **Размер датасета (`len(dataset)`)**. Показывает загрузчику данных, сколько всего сэмплов / картинок доступно для обучения или инференса, чтобы правильно рассчитать количество шагов на эпоху.

4. Фильтрация и Метрики (Сравнение)

В CV/ML вам часто нужно фильтровать предсказания (например, отсекать bounding box'ы с низкой уверенностью).

- `__lt__(self, other)` (<), `__le__` (<=), `__gt__` (>), `__ge__` (>=) — **Реализация Non-Maximum Suppression (NMS) и порогов (Thresholds)**. Используется для фильтрации детекций по метрике Confidence Score или IoU (Intersection over Union). Позволяет быстро сортировать и отбрасывать «слабые» предсказания нейросети.

---